In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the pdfs
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2021', 'creationdate': '2026-02-05T14:51:26+05:30', 'source': '../data/pdf/LINE WALK feb 26.pdf', 'file_path': '../data/pdf/LINE WALK feb 26.pdf', 'total_pages': 10, 'format': 'PDF 1.7', 'title': 'PowerPoint Presentation', 'author': 'admin', 'subject': '', 'keywords': '', 'moddate': '2026-02-05T14:51:26+05:30', 'trapped': '', 'modDate': "D:20260205145126+05'30'", 'creationDate': "D:20260205145126+05'30'", 'page': 0}, page_content='Century Builders\nVendor Code: C256\nLINE WALK\nSINTER PLANT(RMBB#1)\nDate: 05-02-2026\nDOC-CB/EHS/LW-01\nEFFECTIVE DATE- 01-01-2024\nREVISION-00'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2021', 'creationdate': '2026-02-05T14:51:26+05:30', 'source': '../data/pdf/LINE WALK feb 26.pdf', 'file_path': '../data/pdf/LINE WALK feb 26.pdf', 'total_pages': 10, 'format': 'PDF 1.7', 'title': 'PowerPoint Presentation

In [7]:
for idx, pdf_documents in enumerate(pdf_documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", pdf_documents.page_content)
    print("Metadata:\n", pdf_documents.metadata)


Document 1
Content:
 Century Builders
Vendor Code: C256
LINE WALK
SINTER PLANT(RMBB#1)
Date: 05-02-2026
DOC-CB/EHS/LW-01
EFFECTIVE DATE- 01-01-2024
REVISION-00
Metadata:
 {'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2021', 'creationdate': '2026-02-05T14:51:26+05:30', 'source': '../data/pdf/LINE WALK feb 26.pdf', 'file_path': '../data/pdf/LINE WALK feb 26.pdf', 'total_pages': 10, 'format': 'PDF 1.7', 'title': 'PowerPoint Presentation', 'author': 'admin', 'subject': '', 'keywords': '', 'moddate': '2026-02-05T14:51:26+05:30', 'trapped': '', 'modDate': "D:20260205145126+05'30'", 'creationDate': "D:20260205145126+05'30'", 'page': 0}

Document 2
Content:
 Management Line Walk Team
Century Builders
Vendor Code: C256
Audit Team
Anas Khan
Proprietor
Bablu Godsora
Site in-charge
Pappu Kumar
Site in-charge
Sandeep Mahato
Safety Supervisor
Sanjay Kr. Prasad
Site Supervisor
Metadata:
 {'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2

In [8]:
import fitz
import os

pdf_path = "../data/pdf/LINE WALK feb 26.pdf"
output_dir = "extracted_images"
os.makedirs(output_dir, exist_ok=True)

doc = fitz.open(pdf_path)

img_count = 0
for page_index in range(len(doc)):
    page = doc[page_index]
    images = page.get_images(full=True)

    for img in images:
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        image_ext = base_image["ext"]

        img_count += 1
        image_path = f"{output_dir}/image_{img_count}.{image_ext}"
        with open(image_path, "wb") as f:
            f.write(image_bytes)

print(f"Extracted {img_count} images.")


Extracted 29 images.


In [9]:
import fitz

doc = fitz.open("../data/pdf/LINE WALK feb 26.pdf")

for page_num, page in enumerate(doc, start=1):
    blocks = page.get_text("blocks")  # text + images + positions

    print(f"\n--- Page {page_num} ---")
    for block in blocks:
        x0, y0, x1, y1, text, block_type, *_ = block
        print(block_type, text.strip())



--- Page 1 ---
0 Century Builders
Vendor Code: C256
1 LINE WALK
SINTER PLANT(RMBB#1)
2 Date: 05-02-2026
3 DOC-CB/EHS/LW-01
EFFECTIVE DATE- 01-01-2024
REVISION-00

--- Page 2 ---
0 Management Line Walk Team
1 Century Builders
Vendor Code: C256
2 Audit Team
3 Anas Khan
Proprietor
4 Bablu Godsora
Site in-charge
5 Pappu Kumar
Site in-charge
6 Sandeep Mahato
Safety Supervisor
7 Sanjay Kr. Prasad
Site Supervisor

--- Page 3 ---
0 Department
Description of Condition
Root cause
Correction & Immediate Action
1 Taken
Observed on
Severity
Responsible 
Target Date
Status
2 RMBB/Sinter
3 Plant
4 Verified all documents such as valid 
SOP, daily TBM, hot permit and 
hammer clearance , calibration 
certificate etc .found valid and correct
5 Not required 
N/A
05-02-2026
N/A
Supervisor
N/A
N/A
6 Management Line Walk & Finding
Management Line Walk & Finding
7 Century Builders
Vendor Code: C256

--- Page 4 ---
0 Department
Description of Condition
Root cause
Correction & Immediate Action Taken
Observed o

In [10]:
import fitz

doc = fitz.open("../data/pdf/LINE WALK feb 26.pdf")
html = "<html><body>"

for page in doc:
    blocks = page.get_text("blocks")

    for block in blocks:
        x0, y0, x1, y1, text, block_type, *_ = block

        if block_type == 0:
            html += f"<p>{text}</p>"

html += "</body></html>"

with open("output.html", "w", encoding="utf-8") as f:
    f.write(html)


## Json with base64 images (self-contained)

In [12]:
import fitz  # PyMuPDF
import os
import json
import base64

PDF_DIR = "../data/pdf"
OUTPUT_JSON = "pdf_data.json"

all_docs = []

for filename in os.listdir(PDF_DIR):
    if not filename.endswith(".pdf"):
        continue

    path = os.path.join(PDF_DIR, filename)
    doc = fitz.open(path)

    pdf_data = {
        "file_name": filename,
        "pages": []
    }

    for page_num, page in enumerate(doc):
        text = page.get_text()

        images_data = []
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_b64 = base64.b64encode(image_bytes).decode("utf-8")

            images_data.append({
                "image_index": img_index,
                "extension": image_ext,
                "base64": image_b64
            })

        pdf_data["pages"].append({
            "page_number": page_num,
            "text": text,
            "images": images_data
        })

    all_docs.append(pdf_data)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

print("Saved to pdf_data.json")


Saved to pdf_data.json
